# 03 — Provenance Audit

Query the W3C PROV-JSON provenance chain for a generated topology.
Every edge traces to a real neuron or synapse class in the *Drosophila*
male brain connectome — this notebook shows how to inspect that chain.

**Prerequisites:**
- Python 3.10+ with numpy
- Running from inside the `flybrain-connectome` checkout
- Completed [01_generate_visualize.ipynb](01_generate_visualize.ipynb)

## Cell 1 — Load topology and extract provenance

In [ ]:
import sys, os, json
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]
TOOLKIT = HERE / "flybrain-connectome-mcp"
for p in [TOOLKIT, TOOLKIT / "tools"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Load topology
json_path = TOOLKIT / "generated_organisms" / "notebook_demo_m5_a8.json"
with open(json_path, "r", encoding="utf-8") as f:
    topo = json.load(f)

# Extract provenance from all edges
prov_records = []
for e in topo["edges"]:
    prov = e.get("provenance", {})
    if prov:
        prov_records.append({
            "source": e["source"],
            "target": e["target"],
            "source_neuron": prov.get("source_neuron", "?"),
            "source_synapse_class": prov.get("source_synapse_class", "?"),
            "motif_id": prov.get("motif_id", "?"),
            "motif_name": prov.get("motif_name", "?"),
        })

print(f"Total edges: {len(topo['edges'])}")
print(f"Edges with provenance: {len(prov_records)}")
print(f"Provenance coverage: {len(prov_records)/len(topo['edges'])*100:.1f}%")

## Cell 2 — Summarize provenance sources

In [ ]:
from collections import Counter

# Count neurons referenced
neurons = Counter(r["source_neuron"] for r in prov_records if r["source_neuron"] != "?")
print("Neurons referenced by edges:")
for neuron, count in neurons.most_common(10):
    print(f"  {neuron:45s}  {count:3d} edges")

print(f"\nTotal unique neurons: {len(neurons)}")

# Count synapse classes
synapses = Counter(r["source_synapse_class"] for r in prov_records if r["source_synapse_class"] != "?")
print(f"\nSynapse classes referenced:")
for syn, count in synapses.most_common(10):
    print(f"  {syn:45s}  {count:3d} edges")

## Cell 3 — Query specific edge provenance

In [ ]:
# Pick a specific edge and trace its full provenance chain
edge_idx = 5
edge = topo["edges"][edge_idx]
prov = edge.get("provenance", {})

print(f"Edge #{edge_idx}:")
print(f"  Source: {edge['source']}")
print(f"  Target: {edge['target']}")
print(f"  Weight: {edge.get('weight', '?')}")
print(f"  Type  : {edge.get('type', '?')}")
print(f"\nProvenance chain:")
for k, v in prov.items():
    print(f"  {k}: {v}")

## Cell 4 — Check PROV-JSON compliance

In [ ]:
# Check if the topology has a top-level prov_json field
# (requires dataprov package for full W3C PROV-JSON serialization)

has_prov_json = "prov_json" in topo
prov_available = topo.get("prov_json", "") is not None

print(f"Top-level prov_json field: {'present' if has_prov_json else 'absent'}")
print(f"prov_available: {prov_available}")

if has_prov_json and topo["prov_json"]:
    prov_data = topo["prov_json"]
    if isinstance(prov_data, str):
        prov_data = json.loads(prov_data)
    print(f"\nW3C PROV-JSON bundle:")
    print(f"  @context: {prov_data.get('@context', '?')}")
    print(f"  Entities: {len(prov_data.get('entities', []))}")
    print(f"  Activities: {len(prov_data.get('activities', []))}")
    print(f"  derivations: {len(prov_data.get('derivation', []))}")
else:
    print("\n[info] Install dataprov for full W3C PROV-JSON:")
    print("       pip install dataprov")

## Cell 5 — Generate audit report

In [ ]:
report = {
    "topology": str(json_path.name),
    "total_agents": len(topo["agents"]),
    "total_edges": len(topo["edges"]),
    "total_modules": len(topo["modules"]),
    "edges_with_provenance": len(prov_records),
    "provenance_coverage_pct": round(len(prov_records) / len(topo["edges"]) * 100, 1),
    "unique_neurons": len(neurons),
    "unique_synapse_classes": len(synapses),
    "has_prov_json": has_prov_json,
    "top_neurons": [{"neuron": n, "edge_count": c} for n, c in neurons.most_common(5)],
    "top_synapse_classes": [{"class": s, "edge_count": c} for s, c in synapses.most_common(5)],
    "schema_version": topo.get("schema_version", "unknown"),
}

print(json.dumps(report, indent=2))

## What the audit tells you

This audit report is what an EU AI Act auditor would ask for:

1. **Provenance coverage** — what percentage of agent edges trace to
   a real biological source? (Should be 100% for v1.2.1+)
2. **Unique neurons** — how many distinct neural sources feed the topology?
3. **Synapse classes** — what types of biological connections are represented?
4. **W3C PROV-JSON** — is the provenance serialized in the standard
   format that EU AI Act auditors expect?

This is the "查不清" (unauditable) problem solved. Your agent swarm
isn't a black box — every design decision has a traceable biological
prior.

## Next steps

- Run `flybrain profile` for full structural analysis
- Run `flybrain verify` for end-to-end smoke test
- Read `docs/QUICKSTART_5MIN.md` for the full workflow